<a href="https://colab.research.google.com/github/Psyche-1/goit-ds-hw/blob/main/Hw5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
from pathlib import Path

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

In [3]:
base_path = Path('/content/drive/MyDrive/Colab Notebooks/Data/data')
categories = ['walking', 'stairs', 'running', 'idle']

data_frames = []

for category in categories:
    folder_path = base_path / category
    csv_files = folder_path.glob('*.csv')

    for file_path in csv_files:
        temp_df = pd.read_csv(file_path)
        temp_df['activity'] = category
        data_frames.append(temp_df)

df = pd.concat(data_frames, ignore_index=True)

In [4]:
df

,accelerometer_X,accelerometer_Y,accelerometer_Z,activity
0,1.240197,-4.481945,-6.962338,walking
1,3.227384,-8.566454,1.029507,walking
2,-3.600879,-16.721106,-17.707516,walking
3,0.885855,-7.024587,0.981623,walking
4,-1.412579,-2.513912,5.635951,walking
...,...,...,...,...
193855,0.268151,0.086191,9.725247,idle
193856,0.368707,-0.004788,9.777920,idle
193857,0.411803,-0.057461,9.777920,idle
193858,0.469264,-0.076614,9.806650,idle


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 193860 entries, 0 to 193859
Data columns (total 4 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   accelerometer_X  193860 non-null  float64
 1   accelerometer_Y  193860 non-null  float64
 2   accelerometer_Z  193860 non-null  float64
 3   activity         193860 non-null  object 
dtypes: float64(3), object(1)
memory usage: 5.9+ MB


In [6]:
df.describe()

,accelerometer_X,accelerometer_Y,accelerometer_Z
count,193860.000000,193860.000000,193860.000000
mean,1.923550,1.598343,1.804896
std,8.404867,12.474041,7.191590
min,-39.188293,-39.188293,-39.188293
25%,-2.494758,-8.327033,-2.494758
50%,0.248997,-0.009577,0.905008
75%,4.668694,8.671799,7.187394
max,39.188293,39.188293,39.188293


In [7]:
df.duplicated().sum()

np.int64(180673)

In [9]:
df_cleaned = df.drop_duplicates()
df_cleaned

,accelerometer_X,accelerometer_Y,accelerometer_Z,activity
0,1.240197,-4.481945,-6.962338,walking
1,3.227384,-8.566454,1.029507,walking
2,-3.600879,-16.721106,-17.707516,walking
3,0.885855,-7.024587,0.981623,walking
4,-1.412579,-2.513912,5.635951,walking
...,...,...,...,...
191927,-1.599327,0.110133,8.700529,idle
193440,1.000776,4.616021,8.576031,idle
193441,0.718261,4.209007,8.446744,idle
193558,-0.177171,4.692636,8.394072,idle


In [10]:
window_size = 50

axes = ['accelerometer_X', 'accelerometer_Y', 'accelerometer_Z']

df_features = df_cleaned.copy()

grouped = df_features.groupby('activity')

for axis in axes:
    df_features[f'{axis}_mean'] = grouped[axis].transform(lambda x: x.rolling(window_size, min_periods=1).mean())

    df_features[f'{axis}_std'] = grouped[axis].transform(lambda x: x.rolling(window_size, min_periods=1).std().fillna(0))

    df_features[f'{axis}_min'] = grouped[axis].transform(lambda x: x.rolling(window_size, min_periods=1).min())

    df_features[f'{axis}_max'] = grouped[axis].transform(lambda x: x.rolling(window_size, min_periods=1).max())

    df_features[f'{axis}_median'] = grouped[axis].transform(lambda x: x.rolling(window_size, min_periods=1).median())

df_features

,accelerometer_X,accelerometer_Y,accelerometer_Z,activity,accelerometer_X_mean,accelerometer_X_std,accelerometer_X_min,accelerometer_X_max,accelerometer_X_median,accelerometer_Y_mean,accelerometer_Y_std,accelerometer_Y_min,accelerometer_Y_max,accelerometer_Y_median,accelerometer_Z_mean,accelerometer_Z_std,accelerometer_Z_min,accelerometer_Z_max,accelerometer_Z_median
0,1.240197,-4.481945,-6.962338,walking,1.240197,0.000000,1.240197,1.240197,1.240197,-4.481945,0.000000,-4.481945,-4.481945,-4.481945,-6.962338,0.000000,-6.962338,-6.962338,-6.962338
1,3.227384,-8.566454,1.029507,walking,2.233791,1.405153,1.240197,3.227384,2.233791,-6.524199,2.888184,-8.566454,-4.481945,-6.524199,-2.966416,5.651088,-6.962338,1.029507,-2.966416
2,-3.600879,-16.721106,-17.707516,walking,0.288901,3.512124,-3.600879,3.227384,1.240197,-9.923168,6.231354,-16.721106,-4.481945,-8.566454,-7.880116,9.402167,-17.707516,1.029507,-6.962338
3,0.885855,-7.024587,0.981623,walking,0.438139,2.883129,-3.600879,3.227384,1.063026,-9.198523,5.290270,-16.721106,-4.481945,-7.795521,-5.664681,8.863771,-17.707516,1.029507,-2.990358
4,-1.412579,-2.513912,5.635951,walking,0.067996,2.630467,-3.600879,3.227384,0.885855,-7.861601,5.470560,-16.721106,-2.513912,-7.024587,-3.404555,9.190521,-17.707516,5.635951,0.981623
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
191927,-1.599327,0.110133,8.700529,idle,0.184832,0.506864,-2.044648,1.225831,0.351947,2.912499,2.791890,-0.095768,6.473922,4.015076,8.740369,1.040879,5.655105,9.782708,8.853758
193440,1.000776,4.616021,8.576031,idle,0.202741,0.519656,-2.044648,1.225831,0.363919,2.910200,2.790410,-0.095768,6.473922,4.015076,8.731845,1.040436,5.655105,9.782708,8.774749
193441,0.718261,4.209007,8.446744,idle,0.215191,0.524475,-2.044648,1.225831,0.371101,2.898421,2.783512,-0.095768,6.473922,4.015076,8.723801,1.041066,5.655105,9.782708,8.638280
193558,-0.177171,4.692636,8.394072,idle,0.205710,0.527246,-2.044648,1.225831,0.371101,2.879842,2.768022,-0.095768,6.473922,4.015076,8.731654,1.036979,5.655105,9.782708,8.638280


In [13]:
feature_cols = [col for col in df_features.columns if col not in ['accelerometer_X', 'accelerometer_Y', 'accelerometer_Z', 'activity']]

X = df_features[feature_cols]
y = df_features['activity']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [24]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)

rf_pred = rf_model.predict(X_test_scaled)
rf_acc = accuracy_score(y_test, rf_pred)

print(classification_report(y_test, rf_pred))

              precision    recall  f1-score   support

        idle       1.00      1.00      1.00       407
     running       1.00      1.00      1.00      1375
      stairs       1.00      1.00      1.00        88
     walking       1.00      1.00      1.00       768

    accuracy                           1.00      2638
   macro avg       1.00      1.00      1.00      2638
weighted avg       1.00      1.00      1.00      2638



In [25]:
svm_model = SVC(kernel='rbf', C=10.0, random_state=42)
svm_model.fit(X_train_scaled, y_train)

svm_pred = svm_model.predict(X_test_scaled)
svm_acc = accuracy_score(y_test, svm_pred)

print(classification_report(y_test, svm_pred))

              precision    recall  f1-score   support

        idle       1.00      1.00      1.00       407
     running       1.00      1.00      1.00      1375
      stairs       1.00      1.00      1.00        88
     walking       1.00      1.00      1.00       768

    accuracy                           1.00      2638
   macro avg       1.00      1.00      1.00      2638
weighted avg       1.00      1.00      1.00      2638



Модель RandomForestClassifier відпрацювала ідеально.

Модель SVM відпрацювала ідеально.